# 05 — Quick Eval: Stage 1 (easy) vs Baseline

Trained model loaded via vLLM LoRA: base Qwen3-8B + adapter checkpoint.

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

BASE_MODEL = "Qwen/Qwen3-8B"
ADAPTER_DIR = str(PROJECT_ROOT / "artifacts" / "runs" / "curriculum_rl_plus_8b" / "stage1_easy" / "adapter" / "checkpoint-60")

EVAL_DIR = PROJECT_ROOT / "data" / "eval"
OUT_DIR = PROJECT_ROOT / "artifacts" / "eval_stage1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base model: {BASE_MODEL}")
print(f"Adapter: {ADAPTER_DIR}")
print(f"  exists: {Path(ADAPTER_DIR).exists()}")

for d in [1, 3]:
    p = EVAL_DIR / f"eval_d{d}.jsonl"
    print(f"eval_d{d}: {'✓' if p.exists() else 'MISSING'}")

Base model: Qwen/Qwen3-8B
Adapter: /home/yaros/DS-Mag/AI-SelectedTopics/W4/safe_triage_env_project_w3_integrated/artifacts/runs/curriculum_rl_plus_8b/stage1_easy/adapter/checkpoint-60
  exists: True
eval_d1: ✓
eval_d3: ✓


## Run eval

In [2]:
from runtimes.eval_vllm import VLLMEvalConfig, run_vllm_rollouts

EVAL_DIFFICULTIES = [1, 3]

def make_trained_cfg(dataset, tag):
    return VLLMEvalConfig(
        dataset=str(dataset),
        model=BASE_MODEL,
        lora_adapter_dir=ADAPTER_DIR,
        prefer_lora_adapter=True,
        out_traj=str(OUT_DIR / f"{tag}.traj.jsonl"),
        out_metrics=str(OUT_DIR / f"{tag}.metrics.jsonl"),
        max_model_len=4096,
        max_tokens=2048,
        max_cases=100, 
        temperature=0.0,
        top_p=1.0,
        rollout_mode="trajectory",
        enable_thinking=False,
        tensor_parallel_size=1,
        trust_remote_code=True,
    )

def make_baseline_cfg(dataset, tag):
    return VLLMEvalConfig(
        dataset=str(dataset),
        model=BASE_MODEL,
        out_traj=str(OUT_DIR / f"{tag}.traj.jsonl"),
        out_metrics=str(OUT_DIR / f"{tag}.metrics.jsonl"),
        max_model_len=4096,
        max_tokens=2048,
        max_cases=100, 
        temperature=0.0,
        top_p=1.0,
        rollout_mode="trajectory",
        enable_thinking=False,
        tensor_parallel_size=1,
        trust_remote_code=True,
    )

results = {}
for d in EVAL_DIFFICULTIES:
    ds = EVAL_DIR / f"eval_d{d}.jsonl"
    if not ds.exists():
        print(f"d{d}: skipped")
        continue

    print(f"\n{'='*40} d{d} {'='*40}")

    r_t = run_vllm_rollouts(make_trained_cfg(ds, f"trained_d{d}"))
    results[f"trained_d{d}"] = r_t
    print(f"  Trained:  success={r_t.summary['success_rate']:.0%}  "
          f"reward={r_t.summary['avg_total_reward']:.2f}  "
          f"tools={r_t.summary['avg_tool_calls']:.1f}  "
          f"ev_cov={r_t.summary['avg_evidence_coverage']:.2f}")

    r_b = run_vllm_rollouts(make_baseline_cfg(ds, f"baseline_d{d}"))
    results[f"baseline_d{d}"] = r_b
    print(f"  Baseline: success={r_b.summary['success_rate']:.0%}  "
          f"reward={r_b.summary['avg_total_reward']:.2f}  "
          f"tools={r_b.summary['avg_tool_calls']:.1f}  "
          f"ev_cov={r_b.summary['avg_evidence_coverage']:.2f}")


======================================== d1 ========================================


[2026-03-14 14:50:53] INFO api.py:441: Resolved model path: Qwen/Qwen3-8B
[2026-03-14 14:50:53] INFO api.py:443: Using LoRA adapter: /home/yaros/DS-Mag/AI-SelectedTopics/W4/safe_triage_env_project_w3_integrated/artifacts/runs/curriculum_rl_plus_8b/stage1_easy/adapter/checkpoint-60


INFO 03-14 14:50:54 [utils.py:238] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'disable_log_stats': True, 'enable_lora': True, 'max_loras': 4, 'max_lora_rank': 64, 'model': 'Qwen/Qwen3-8B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-14 14:50:55 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 03-14 14:50:55 [model.py:1554] Using max model len 4096
INFO 03-14 14:50:55 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 03-14 14:50:55 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 03-14 14:50:57 [interface.py:472] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
WARNING 03-14 14:50:57 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: WSL is detected and NVML is not compatible with fork
(EngineCore_DP0 pid=23195) INFO 03-14 14:51:03 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None,

...

(EngineCore_DP0 pid=32157) 2026-03-14 15:39:39,549 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=32157) 2026-03-14 15:39:39,880 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 21.31it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:01<00:00, 32.14it/s]


(EngineCore_DP0 pid=32157) INFO 03-14 15:39:44 [gpu_model_runner.py:5360] Graph capturing finished in 5 secs, took -1.77 GiB
(EngineCore_DP0 pid=32157) INFO 03-14 15:39:44 [core.py:282] init engine (profile, create kv cache, warmup model) took 9.62 seconds
(EngineCore_DP0 pid=32157) INFO 03-14 15:39:46 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 03-14 15:39:46 [llm.py:388] Supported tasks: ['generate']


vLLM trajectory rollout:   0%|          | 0/100 [00:00<?, ?it/s]

[rank0]:[W314 15:40:24.424447103 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


  Baseline: success=0%  reward=-1.08  tools=0.6  ev_cov=0.25


## Results

In [3]:
import pandas as pd

rows = []
for d in EVAL_DIFFICULTIES:
    for label, key in [("Baseline", "baseline"), ("Trained", "trained")]:
        r = results.get(f"{key}_d{d}")
        if r is None:
            continue
        s = r.summary
        rows.append({
            "model": label, "difficulty": d,
            "success_rate": s.get("success_rate"),
            "avg_reward": s.get("avg_total_reward"),
            "avg_tool_calls": s.get("avg_tool_calls"),
            "evidence_coverage": s.get("avg_evidence_coverage"),
            "violations": s.get("avg_policy_violations"),
        })

df = pd.DataFrame(rows)
print("=== Results ===")
display(df.set_index(["difficulty", "model"]))

print("\n=== Failure Reasons ===")
for d in EVAL_DIFFICULTIES:
    for label, key in [("Baseline", "baseline"), ("Trained", "trained")]:
        r = results.get(f"{key}_d{d}")
        if r:
            print(f"  d{d} {label}: {r.summary.get('failure_reasons', {})}")

=== Results ===


success_rate  avg_reward  avg_tool_calls  \
difficulty model                                                
1          Baseline          0.00     -1.0904            0.44   
           Trained           0.20     -0.9762            3.47   
3          Baseline          0.00     -1.0794            0.58   
           Trained           0.19     -1.0481            4.27   

                     evidence_coverage  violations  
difficulty model                                    
1          Baseline               0.14        0.03  
           Trained                0.78        0.73  
3          Baseline               0.25        0.00  
           Trained                0.81        0.79


=== Failure Reasons ===
  d1 Baseline: {'actions_exhausted': 100}
  d1 Trained: {'premature_finish': 5, 'actions_exhausted': 20, 'hallucinated_slot': 28, 'advice_pack_not_allowed': 3, 'hallucinated_advice_pack': 10, 'max_steps_exceeded': 5, 'wrong_final_decision': 8, 'slot_not_found': 1}
  d3 Baseline: {'actions_exhausted': 100}
  d3 Trained: {'wrong_final_decision': 15, 'actions_exhausted': 21, 'hallucinated_slot': 28, 'max_steps_exceeded': 4, 'hallucinated_advice_pack': 6, 'slot_not_found': 2, 'premature_finish': 5}
